# HLLD ML — Training Pipeline Walkthrough

This notebook documents the full ML pipeline for learning the HLLD total pressure $p^*_{\rm tot}$ from primitive Riemann states. It covers:

1. [Project structure](#1-project-structure)
2. [Data generation](#2-data-generation)
3. [Preprocessing](#3-preprocessing)
4. [Dataset & DataLoader](#4-dataset--dataloader)
5. [Model architecture](#5-model-architecture)
6. [Loss function](#6-loss-function)
7. [Training loop](#7-training-loop)
8. [Evaluation](#8-evaluation)
9. [Bugs found and fixed](#9-bugs-found-and-fixed)
10. [Results](#10-results)

In [ ]:
import os, sys
# Run from the project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Working directory:', os.getcwd())

In [ ]:
import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

CONFIG = 'configs/default-mignone.yaml'
with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

config_stem = os.path.splitext(os.path.basename(CONFIG))[0]
cfg['data']['split_dir']   = os.path.join('data/splits',    config_stem)
cfg['_checkpoint_path']    = os.path.join('checkpoints',    f'{config_stem}.pt')
cfg['_results_dir']        = os.path.join('results',        config_stem)
cfg['_config_stem']        = config_stem
print(f'Config: {CONFIG}  |  stem: {config_stem}')

---
## 1  Project structure

```
HLLD/
├── configs/
│   ├── default-mignone.yaml   ← Mignone et al. (2022) 1-D shock-tube regime
│   └── default.yaml
├── src/
│   ├── data/
│   │   ├── generate.py        ← sample random Riemann problems, compute p*
│   │   ├── preprocess.py      ← normalise inputs & targets, split train/val/test
│   │   └── dataset.py         ← PyTorch Dataset / DataLoader
│   ├── models/
│   │   ├── network.py         ← PressureNet (MLP + Softplus output)
│   │   └── losses.py          ← PhysicsInformedLoss, RelativeMSELoss
│   └── training/
│       ├── train.py           ← training loop, checkpointing
│       └── evaluate.py        ← test-set metrics, inverse-transform
├── data/
│   ├── raw/                   ← generated .npz datasets
│   ├── splits/<stem>/         ← train / val / test splits
│   └── processed/<stem>/      ← normalisation statistics
├── checkpoints/               ← best model per config
└── results/                   ← predictions and plots
```

### Full pipeline
```bash
python -m src.data.generate   --config configs/default-mignone.yaml
python -m src.data.preprocess --config configs/default-mignone.yaml
python -m src.training.train  --config configs/default-mignone.yaml
python -m src.training.evaluate --config configs/default-mignone.yaml
```

All outputs (splits, checkpoints, results) are automatically namespaced by the config filename stem, so multiple configs can coexist.

---
## 2  Data generation

`src/data/generate.py` samples random Riemann left/right states on the GPU (or CPU/MPS) and calls the analytical HLLD solver to compute the exact $p^*_{\rm tot}$.

### Feature vector  (N × 15)
| idx | feature |
|-----|---------|
| 0–6 | $\rho_L,\, v_{x,L},\, v_{y,L},\, v_{z,L},\, p_L,\, B_{y,L},\, B_{z,L}$ |
| 7–13 | same for right state |
| 14 | $B_x$ (shared, divergence-free) |

### Sampling strategy
- **Density** — log-uniform: `rho = 10^lrho`, `lrho ~ U(-0.9, 2.0)`
- **Velocity** — sampled via Lorentz factor `W ~ U(1, 25)` + random direction on unit sphere → sub-luminal by construction
- **Pressure** — via `(T, rho)` through the hybrid EOS
- **Magnetic fields** — `Bx` uniform; transverse `(|Bt|, φ)` for isotropy

In [ ]:
# Inspect the raw dataset
raw_path = cfg['data']['raw_path']
d = np.load(raw_path)
inputs, targets = d['inputs'], d['targets']

print(f'inputs  shape : {inputs.shape}   dtype: {inputs.dtype}')
print(f'targets shape : {targets.shape}   dtype: {targets.dtype}')
print(f'p_tot*  range : [{targets.min():.4f}, {targets.max():.4f}]')
print(f'non-finite targets: {(~np.isfinite(targets)).sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].hist(targets, bins=80, color='steelblue', edgecolor='none')
axes[0].set_xlabel('$p^*_{\\rm tot}$ (raw)')
axes[0].set_ylabel('count')
axes[0].set_title('Target distribution (linear scale)')

axes[1].hist(np.log10(targets[targets > 0]), bins=80, color='darkorange', edgecolor='none')
axes[1].set_xlabel('$\\log_{10}(p^*_{\\rm tot})$')
axes[1].set_title('Target distribution (log scale)')

plt.tight_layout()
plt.show()

---
## 3  Preprocessing

`src/data/preprocess.py` does three things:

1. **Sanity-check targets** — drops any `nan`/`inf` produced by degenerate HLLD configurations.
2. **Normalise inputs** — standard (zero-mean, unit-variance) by default; `minmax` and `log` also supported.
3. **Normalise targets** — `log10` then standardise. This is the natural choice because $p^*_{\rm tot}$ is always positive and spans orders of magnitude in the Mignone regime.

Normalisation statistics are saved to `data/processed/<stem>/norm_stats.npz` so that `evaluate.py` can invert the transform.

### ⚠️ Bug fixed — targets were not normalised
The original code passed raw `y` to `split()` while only normalising `X`.  
With $p^*_{\rm tot}$ spanning large values and the network initialised near zero, the MSE blew up to **~1.9 × 10⁹** per sample on the first epoch — making it impossible to tell if the model was learning anything meaningful.

In [ ]:
# Inspect normalised splits
split_dir = cfg['data']['split_dir']
train_data = np.load(os.path.join(split_dir, 'train.npz'))
val_data   = np.load(os.path.join(split_dir, 'val.npz'))
test_data  = np.load(os.path.join(split_dir, 'test.npz'))

y_train = train_data['targets']
print(f'Train targets  mean={y_train.mean():.4f}  std={y_train.std():.4f}')
print(f'               min={y_train.min():.4f}    max={y_train.max():.4f}')

stats_path = os.path.join('data/processed', config_stem, 'norm_stats.npz')
stats = np.load(stats_path)
print(f'\ny_mu={float(stats["y_mu"]):.4f}  y_sigma={float(stats["y_sigma"]):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].hist(y_train, bins=80, color='steelblue', edgecolor='none')
axes[0].set_xlabel('normalised $\\log_{10}(p^*_{\\rm tot})$')
axes[0].set_ylabel('count')
axes[0].set_title('Normalised target (train)')

x_train = train_data['inputs']
feature_names = [
    r'$\rho_L$', r'$v_{x,L}$', r'$v_{y,L}$', r'$v_{z,L}$', r'$p_L$', r'$B_{y,L}$', r'$B_{z,L}$',
    r'$\rho_R$', r'$v_{x,R}$', r'$v_{y,R}$', r'$v_{z,R}$', r'$p_R$', r'$B_{y,R}$', r'$B_{z,R}$',
    r'$B_x$'
]
axes[1].boxplot(x_train, vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.5),
                medianprops=dict(color='navy'),
                flierprops=dict(marker='.', markersize=1, alpha=0.2),
                labels=feature_names)
axes[1].axhline(0, color='gray', lw=0.8, ls='--')
axes[1].set_title('Normalised input features (train)')
axes[1].tick_params(axis='x', rotation=45, labelsize=7)

plt.tight_layout()
plt.show()

---
## 4  Dataset & DataLoader

`HLLDDataset` is a standard `torch.utils.data.Dataset` that loads an `.npz` split and returns `(X, y)` tensors.

### ⚠️ Bug fixed — `pin_memory=True` unconditionally
`pin_memory` accelerates GPU data transfer but **requires CUDA**. On CPU and MPS it wastes memory and can silently degrade performance. Fixed: `pin_memory = (device == 'cuda')`.

### ⚠️ Bug fixed — `num_workers=4` on macOS  
PyTorch multiprocessing workers on macOS use `torch_shm_manager` for shared memory between processes. On many macOS setups the binary doesn't have execute permission, causing:  
```
RuntimeError: torch_shm_manager ... execl failed: Permission denied
```  
Fixed: `num_workers = 4 if device == 'cuda' else 0`.

In [ ]:
from src.data.dataset import get_dataloaders

device_str = cfg['training'].get('device', 'cpu')
loaders = get_dataloaders(
    cfg['data']['split_dir'],
    batch_size=cfg['training']['batch_size'],
    device=device_str,
)

for split, loader in loaders.items():
    X_batch, y_batch = next(iter(loader))
    print(f'{split:5s}  batches={len(loader):5d}  '
          f'X={tuple(X_batch.shape)}  y={tuple(y_batch.shape)}  '
          f'pin_memory={loader.pin_memory}')

---
## 5  Model architecture

`PressureNet` is a fully-connected MLP:

```
Input (15) → Linear → SiLU → Linear → SiLU → Linear → SiLU → Linear → Softplus → Output (1)
              128              128              64
```

### ⚠️ Bug fixed — no output activation
The original network had no activation on the final layer, so it could produce **negative** $p^*_{\rm tot}$ — which is physically impossible. `Softplus(x) = log(1 + e^x)` was added as the output activation: it is smooth, always positive, and approximates a linear pass-through for large positive inputs (unlike `ReLU` which saturates the gradient at zero).

### ⚠️ Bug fixed — wrong `n_input` default (16 instead of 15)
Both `train.py` and `evaluate.py` defaulted to `n_input=16` when the feature vector has **15** elements. If `n_input` was absent from the config, the model built in training (16 inputs) would be incompatible with the one built in evaluation, causing a checkpoint load error.

In [ ]:
from src.models.network import PressureNet

model = PressureNet(
    n_input=cfg['model'].get('n_input', 15),
    hidden=cfg['model'].get('hidden', [128, 128, 64]),
    activation=cfg['model'].get('activation', 'silu'),
)
print(model)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTrainable parameters: {n_params:,}')

In [ ]:
# Verify output is always positive (Softplus guarantee)
x_test = torch.randn(10000, 15)
with torch.no_grad():
    out = model(x_test)
print(f'Output min (random init): {out.min().item():.6f}  — should be > 0')
assert (out > 0).all(), 'Found non-positive outputs!'

---
## 6  Loss function

`PhysicsInformedLoss` combines a standard MSE data loss with a positivity penalty:

$$\mathcal{L} = \underbrace{\text{MSE}(\hat{y}, y)}_{\text{data loss}} + \lambda_{\text{phys}} \underbrace{\langle \text{ReLU}(-\hat{y})^2 \rangle}_{\text{positivity penalty}}$$

With the `Softplus` output the positivity penalty is always zero — it is kept as a safeguard.

`RelativeMSELoss` is available as an alternative when $p^*_{\rm tot}$ spans orders of magnitude and equal relative (rather than absolute) error is desired.

### ⚠️ Bug fixed — custom `MSELoss` class
The original file defined a `class MSELoss(nn.Module)` that re-implemented `torch.mean((pred - target) ** 2)` — identical to `nn.MSELoss()` but heavier. It was removed and `PhysicsInformedLoss` now uses `nn.MSELoss()` directly. The re-export from `src/models/__init__.py` was also removed, which had been causing an `ImportError` at startup.

In [ ]:
from src.models.losses import PhysicsInformedLoss, RelativeMSELoss

criterion = PhysicsInformedLoss(lambda_phys=cfg['training'].get('lambda_phys', 0.1))

# Confirm positivity penalty is 0 with Softplus output
pred   = torch.tensor([[1.2], [0.3], [2.1]])
target = torch.tensor([[1.0], [0.5], [1.8]])
loss   = criterion(pred, target)
mse    = torch.nn.functional.mse_loss(pred, target)
print(f'loss={loss.item():.6f}  mse={mse.item():.6f}  diff={abs(loss.item()-mse.item()):.2e}')

---
## 7  Training loop

`src/training/train.py` runs Adam with `ReduceLROnPlateau`.

Key hyperparameters from `default-mignone.yaml`:

| param | value | yaml key |
|-------|-------|----------|
| epochs | 200 | `training.epochs` |
| batch size | 256 | `training.batch_size` |
| learning rate | 1e-3 | `training.lr` |
| weight decay | 1e-5 | `training.weight_decay` |
| LR patience | 10 | `training.lr_patience` |
| LR factor | 0.5 | `training.lr_factor` |
| λ_phys | 0.1 | `training.lambda_phys` |

### ⚠️ Bug fixed — hardcoded checkpoint path
The original code always saved to `checkpoints/best_model.pt`, so running two configs in sequence silently **overwrote** the first checkpoint. Fixed: checkpoint is now `checkpoints/<config_stem>.pt`.

### ⚠️ Bug fixed — `split_dir` read from config yaml
`train.py` used `cfg["data"]["split_dir"]` directly. After we changed `preprocess.py` to write splits to `data/splits/<stem>/`, the hardcoded yaml value was stale. Fixed: both scripts derive the path from the config filename.

### ⚠️ Bug fixed — scheduler params hardcoded
`ReduceLROnPlateau(patience=10, factor=0.5)` was not configurable. Both are now read from the yaml.

In [ ]:
# Plot training curves if a log exists, or parse stdout output
# Paste your training output below as a string for quick plotting

log_text = """
Epoch    1  train=0.569623  val=0.520109
Epoch   10  train=0.485399  val=0.483321
Epoch   20  train=0.477989  val=0.473431
Epoch   30  train=0.473481  val=0.468683
Epoch   40  train=0.471060  val=0.464039
Epoch   50  train=0.469226  val=0.464603
Epoch   60  train=0.467792  val=0.465134
Epoch   70  train=0.466610  val=0.461201
Epoch   80  train=0.465843  val=0.461701
Epoch   90  train=0.465026  val=0.459453
Epoch  100  train=0.460232  val=0.455938
"""

epochs, train_losses, val_losses = [], [], []
for line in log_text.strip().splitlines():
    parts = line.split()
    if len(parts) >= 4:
        epochs.append(int(parts[1]))
        train_losses.append(float(parts[2].split('=')[1]))
        val_losses.append(float(parts[3].split('=')[1]))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs, train_losses, 'o-', label='train', color='steelblue')
ax.plot(epochs, val_losses,   's--', label='val',   color='darkorange')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE (normalised log$_{10}$ space)')
ax.set_title('Training curves — default-mignone')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Reading the loss

The MSE is computed in **normalised log₁₀ space**: targets were transformed as

$$y_{\rm norm} = \frac{\log_{10}(p^*_{\rm tot}) - \mu_{\log}}{\sigma_{\log}}$$

so an MSE of ~0.46 means predictions are $\sqrt{0.46} \approx 0.68\sigma$ off in log₁₀ space, which translates to a typical factor of $10^{0.68 \cdot \sigma_{\log}}$ relative error in physical units. After training completes, `evaluate.py` inverts the transform and reports metrics in physical pressure units.

---
## 8  Evaluation

`src/training/evaluate.py` loads the best checkpoint, runs the test set, inverts the normalisation, and reports metrics in physical units.

### ⚠️ Bug fixed — results directory never created
`np.savez('results/test_predictions.npz', ...)` was called without `os.makedirs` first, causing a `FileNotFoundError`. Fixed and namespaced to `results/<stem>/`.

### ⚠️ Bug fixed — `split_dir` not derived from config stem  
Same issue as in `train.py` — `evaluate.py` still read `cfg["data"]["split_dir"]` from the yaml after we changed the path convention. Fixed.

### ⚠️ Bug fixed — hardcoded checkpoint path  
Always loaded `checkpoints/best_model.pt`, so evaluating config B after training config A would silently evaluate the wrong model. Fixed: loads `checkpoints/<stem>.pt`.

In [ ]:
# Run evaluation (requires a trained checkpoint)
checkpoint_path = cfg['_checkpoint_path']
if not os.path.exists(checkpoint_path):
    print(f'No checkpoint found at {checkpoint_path} — run training first.')
else:
    from src.data.dataset import HLLDDataset
    from src.models.network import PressureNet

    device = torch.device(cfg['training'].get('device', 'cpu'))
    model  = PressureNet(
        n_input=cfg['model'].get('n_input', 15),
        hidden=cfg['model'].get('hidden', [128, 128, 64]),
        activation=cfg['model'].get('activation', 'silu'),
    ).to(device)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()

    ds = HLLDDataset(os.path.join(cfg['data']['split_dir'], 'test.npz'))
    X  = ds.X.to(device)
    y_norm_true = ds.y.numpy().flatten()

    with torch.no_grad():
        y_norm_pred = model(X).cpu().numpy().flatten()

    # Inverse transform
    stats   = np.load(os.path.join('data/processed', config_stem, 'norm_stats.npz'))
    y_mu    = float(stats['y_mu'])
    y_sigma = float(stats['y_sigma'])
    y_true  = 10.0 ** (y_norm_true * y_sigma + y_mu)
    y_pred  = 10.0 ** (y_norm_pred * y_sigma + y_mu)
    rel_err = np.abs(y_pred - y_true) / (np.abs(y_true) + 1e-12)

    print(f'Test MSE        : {np.mean((y_pred - y_true)**2):.6e}')
    print(f'Test RMSE       : {np.sqrt(np.mean((y_pred - y_true)**2)):.6e}')
    print(f'Mean  rel error : {rel_err.mean():.4%}')
    print(f'Median rel error: {np.median(rel_err):.4%}')
    print(f'Max   rel error : {rel_err.max():.4%}')

---
## 10  Results

In [ ]:
checkpoint_path = cfg['_checkpoint_path']
if not os.path.exists(checkpoint_path):
    print('Run training first.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    # 1. Predicted vs true
    ax = axes[0]
    sc = ax.scatter(y_true, y_pred, s=1, alpha=0.15, c=np.log10(rel_err + 1e-8),
                    cmap='plasma', rasterized=True)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, 'w--', lw=1)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('True $p^*_{\\rm tot}$')
    ax.set_ylabel('Predicted $p^*_{\\rm tot}$')
    ax.set_title('Predicted vs True')
    plt.colorbar(sc, ax=ax, label='$\\log_{10}$(rel err)')

    # 2. Relative error distribution
    ax = axes[1]
    ax.hist(rel_err, bins=100, color='steelblue', edgecolor='none', log=True)
    ax.axvline(np.median(rel_err), color='darkorange', lw=1.5, label=f'median={np.median(rel_err):.3%}')
    ax.axvline(rel_err.mean(),     color='crimson',    lw=1.5, ls='--', label=f'mean={rel_err.mean():.3%}')
    ax.set_xlabel('Relative error')
    ax.set_ylabel('count (log)')
    ax.set_title('Relative error distribution')
    ax.legend(fontsize=9)

    # 3. Relative error vs true value
    ax = axes[2]
    ax.scatter(y_true, rel_err, s=1, alpha=0.1, color='steelblue', rasterized=True)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('True $p^*_{\\rm tot}$')
    ax.set_ylabel('Relative error')
    ax.set_title('Error vs pressure')
    ax.axhline(np.median(rel_err), color='darkorange', lw=1.2, ls='--')

    plt.tight_layout()
    os.makedirs(cfg['_results_dir'], exist_ok=True)
    fig.savefig(os.path.join(cfg['_results_dir'], 'evaluation_plots.png'), dpi=150)
    plt.show()

---
## 9  Bugs found and fixed — summary

| # | File | Bug | Effect | Fix |
|---|------|-----|--------|-----|
| 1 | `preprocess.py` | Targets `y` not normalised | MSE ~1.9 × 10⁹ per sample on first epoch | `log10` + standardise `y`; save `y_mu`, `y_sigma` to `norm_stats.npz` |
| 2 | `train.py` / `evaluate.py` | Checkpoint hardcoded to `best_model.pt` | Multiple configs silently overwrite each other's checkpoint | Save to `checkpoints/<config_stem>.pt` |
| 3 | `train.py` / `evaluate.py` | `split_dir` read from yaml (stale after renaming convention) | `FileNotFoundError` at DataLoader init | Derive from config filename: `data/splits/<stem>/` |
| 4 | `evaluate.py` | `n_input` default = 16 (not 15) | Shape mismatch when loading checkpoint | Change default to 15 |
| 5 | `evaluate.py` | `results/` dir never created | `FileNotFoundError` on `np.savez` | `os.makedirs(results_dir, exist_ok=True)` |
| 6 | `evaluate.py` | Results hardcoded to `results/test_predictions.npz` | Multiple configs overwrite each other | Namespace to `results/<stem>/` |
| 7 | `generate.py` | `--output` defaulted to `data/raw/dataset.npz` regardless of config | Generated file path didn't match `raw_path` in yaml | Default to `cfg["data"]["raw_path"]` |
| 8 | `preprocess.py` | Norm stats saved to shared `data/processed/norm_stats.npz` | Multiple configs overwrite each other's stats | Namespace to `data/processed/<stem>/` |
| 9 | `network.py` | No output activation | Network can predict negative pressure | Added `nn.Softplus()` as final layer |
| 10 | `dataset.py` | `pin_memory=True` unconditionally | Wastes memory / can error on CPU/MPS | `pin_memory = (device == 'cuda')` |
| 11 | `dataset.py` | `num_workers=4` on macOS | `torch_shm_manager` permission error, training never starts | `num_workers = 4 if device == 'cuda' else 0` |
| 12 | `losses.py` | Custom `MSELoss` class re-implemented `nn.MSELoss` | Stale export in `__init__.py` caused `ImportError` | Remove; use `nn.MSELoss()` directly |
| 13 | `train.py` | Scheduler `patience`/`factor` hardcoded | Not tunable without editing source | Expose as `lr_patience` / `lr_factor` in yaml |